# 03d — Advanced Product & Category Analytics (Q17–Q21)

> **Session 3D: Volatility, seasonality, and trend detection**

---

## Setup

In [ ]:
# ── Connect to database ─────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from shared_setup import get_connection
conn = get_connection()

---
## Q17 — Revenue Volatility per Product

STDDEV and Coefficient of Variation for each product

In [ ]:
# Q17 — Revenue Volatility per Product
# TODO: Session 3D
product_revenue_volatility = conn.execute('''
WITH product_monthly AS (
  SELECT
    f.product_key,
    p.product_name,
    date_trunc('month', d.full_date) AS month_start,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Date d ON f.date_key = d.date_key
  JOIN Dim_Product p ON f.product_key = p.product_key
  GROUP BY 1, 2, 3
)
SELECT
  product_key,
  product_name,
  STDDEV_SAMP(monthly_revenue) AS revenue_stddev,
  AVG(monthly_revenue) AS revenue_avg,
  CASE
    WHEN AVG(monthly_revenue) = 0 THEN NULL
    ELSE STDDEV_SAMP(monthly_revenue) / AVG(monthly_revenue)
  END AS coefficient_of_variation
FROM product_monthly
GROUP BY 1, 2
ORDER BY coefficient_of_variation DESC NULLS LAST;
''').fetchdf()

print(product_revenue_volatility.head(50))

---
## Q18 — Trending Categories

Compare recent 3-month performance vs historical average

In [ ]:
# Q18 — Trending Categories
# TODO: Session 3D
trending_categories = conn.execute('''
WITH monthly AS (
  SELECT
    c.category_key,
    c.category_name,
    date_trunc('month', d.full_date) AS month_start,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Date d       ON f.date_key = d.date_key
  JOIN Dim_Category c  ON f.category_key = c.category_key
  GROUP BY 1, 2, 3
),
max_month AS (
  SELECT MAX(month_start) AS max_month_start
  FROM monthly
),
recent_3_months AS (
  SELECT
    m.category_key,
    m.category_name,
    m.monthly_revenue
  FROM monthly m
  CROSS JOIN max_month mm
  WHERE m.month_start >= (mm.max_month_start - INTERVAL '2 months')
    AND m.month_start <= mm.max_month_start
),
historical_months AS (
  SELECT
    m.category_key,
    m.category_name,
    m.monthly_revenue
  FROM monthly m
  CROSS JOIN max_month mm
  WHERE m.month_start < (mm.max_month_start - INTERVAL '2 months')
),
recent_avg AS (
  SELECT
    category_key,
    category_name,
    AVG(monthly_revenue) AS recent_avg_revenue
  FROM recent_3_months
  GROUP BY 1, 2
),
historical_avg AS (
  SELECT
    category_key,
    category_name,
    AVG(monthly_revenue) AS historical_avg_revenue
  FROM historical_months
  GROUP BY 1, 2
)
SELECT
  r.category_name,
  r.recent_avg_revenue,
  h.historical_avg_revenue,
  (r.recent_avg_revenue - h.historical_avg_revenue) AS trend_diff,
  r.recent_avg_revenue / NULLIF(h.historical_avg_revenue, 0) AS trend_ratio
FROM recent_avg r
LEFT JOIN historical_avg h
  ON r.category_key = h.category_key
ORDER BY trend_ratio DESC NULLS LAST;
''').fetchdf()

print(trending_categories.head(30))

---
## Q19 — Seasonality Analysis (YoY Same-Month)

Same-month comparison across years using LAG()

In [ ]:
# Q19 — Seasonality Analysis (YoY Same-Month)
# TODO: Session 3D

seasonality_yoy_same_month = conn.execute('''
WITH monthly_cat AS (
  SELECT
    c.category_key,
    c.category_name,
    CAST(strftime(d.full_date, '%m') AS INTEGER) AS month_num,  -- 1..12
    CAST(strftime(d.full_date, '%Y') AS INTEGER) AS year,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Category c
    ON f.category_key = c.category_key
  JOIN Dim_Date d
    ON f.date_key = d.date_key
  GROUP BY 1, 2, 3, 4
)
SELECT
  category_name,
  year,
  month_num,
  monthly_revenue,
  LAG(monthly_revenue) OVER (
    PARTITION BY category_key, month_num
    ORDER BY year
  ) AS prev_year_same_month_revenue,
  (monthly_revenue
    - LAG(monthly_revenue) OVER (
        PARTITION BY category_key, month_num
        ORDER BY year
      )
  ) AS yoy_diff,
  (monthly_revenue
    - LAG(monthly_revenue) OVER (
        PARTITION BY category_key, month_num
        ORDER BY year
      )
  ) / NULLIF(
      LAG(monthly_revenue) OVER (
        PARTITION BY category_key, month_num
        ORDER BY year
      ),
      0
  ) AS yoy_growth_ratio
FROM monthly_cat
ORDER BY category_name, month_num, year;
''').fetchdf()

print(seasonality_yoy_same_month.head(50))

---
## Q20 — Profit Consistency Across Time

Monthly margin STDDEV and swing analysis per product

In [ ]:
# Q20 — Profit Consistency Across Time
# TODO: Session 3D
profit_consistency = conn.execute('''
WITH product_monthly AS (
  SELECT
    f.product_key,
    p.product_name,
    date_trunc('month', d.full_date) AS month_start,
    SUM(f.profit_amount) AS profit_sum,
    SUM(f.gross_amount)  AS gross_sum,
    SUM(f.profit_amount) / NULLIF(SUM(f.gross_amount), 0) AS margin_ratio
  FROM Fact_Order_Line f
  JOIN Dim_Product p ON f.product_key = p.product_key
  JOIN Dim_Date d    ON f.date_key = d.date_key
  GROUP BY 1, 2, 3
),
with_deltas AS (
  SELECT
    *,
    LAG(margin_ratio) OVER (
      PARTITION BY product_key
      ORDER BY month_start
    ) AS prev_margin_ratio,
    margin_ratio - LAG(margin_ratio) OVER (
      PARTITION BY product_key
      ORDER BY month_start
    ) AS margin_swing_ratio
  FROM product_monthly
)
SELECT
  product_key,
  product_name,

  STDDEV_SAMP(margin_ratio) AS monthly_margin_stddev_ratio,
  AVG(margin_ratio) AS avg_monthly_margin_ratio,

  MAX(margin_ratio) AS max_monthly_margin_ratio,
  MIN(margin_ratio) AS min_monthly_margin_ratio,
  (MAX(margin_ratio) - MIN(margin_ratio)) AS swing_range_ratio,

  AVG(ABS(margin_swing_ratio)) AS avg_abs_month_to_month_swing_ratio
FROM with_deltas
GROUP BY 1, 2
ORDER BY monthly_margin_stddev_ratio DESC NULLS LAST;
''').fetchdf()

print(profit_consistency.head(50))

---
## Q21 — Consecutive Revenue Decline Detection

Identify products with 3+ months of consecutive decline

In [ ]:
# Q21 — Consecutive Revenue Decline Detection
# TODO: Session 3D
consecutive_decline_products = conn.execute('''
WITH monthly AS (
  SELECT
    f.product_key,
    p.product_name,
    date_trunc('month', d.full_date) AS month_start,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Product p ON f.product_key = p.product_key
  JOIN Dim_Date d ON f.date_key = d.date_key
  GROUP BY 1, 2, 3
),
flags AS (
  SELECT
    product_key,
    product_name,
    month_start,
    monthly_revenue,
    LAG(monthly_revenue) OVER (
      PARTITION BY product_key
      ORDER BY month_start
    ) AS prev_revenue,
    CASE
      WHEN LAG(monthly_revenue) OVER (PARTITION BY product_key ORDER BY month_start) IS NULL THEN 0
      WHEN monthly_revenue < LAG(monthly_revenue) OVER (PARTITION BY product_key ORDER BY month_start) THEN 1
      ELSE 0
    END AS is_decline,
    CASE
      WHEN LAG(monthly_revenue) OVER (PARTITION BY product_key ORDER BY month_start) IS NULL THEN 1
      WHEN monthly_revenue >= LAG(monthly_revenue) OVER (PARTITION BY product_key ORDER BY month_start) THEN 1
      ELSE 0
    END AS is_reset
  FROM monthly
),
segmented AS (
  SELECT
    *,
    SUM(is_reset) OVER (
      PARTITION BY product_key
      ORDER BY month_start
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS seg_id
  FROM flags
),
streaks AS (
  SELECT
    product_key,
    product_name,
    seg_id,
    SUM(is_decline) AS consecutive_decline_months,
    MIN(CASE WHEN is_decline = 1 THEN month_start END) AS streak_start_month,
    MAX(CASE WHEN is_decline = 1 THEN month_start END) AS streak_end_month
  FROM segmented
  GROUP BY 1, 2, 3
)
SELECT
  product_key,
  product_name,
  consecutive_decline_months,
  streak_start_month,
  streak_end_month
FROM streaks
WHERE consecutive_decline_months >= 3
ORDER BY consecutive_decline_months DESC, product_name, streak_start_month;
''').fetchdf()

print(consecutive_decline_products.head(50))